# TwelveLabs Pegasus 1.5 on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Pegasus 1.5 reads a video and answers in text. It takes in what it sees, what is said and
what is written on screen in a single pass; the clip used here is silent, so the cells
below exercise the pictures and the on-screen text (section 4) but not speech. It has two
modes. In the default mode you ask a question about the
video and get prose or JSON back. In `time_based_metadata` mode you describe the moments
you care about and it returns each one with its start and end time, with no prompt at
all (section 5).

| Model | `bedrock-runtime` ID | Inference profiles | Input | Max input | Max output | Notes |
|---|---|---|---|---|---|---|
| Pegasus 1.5 | `twelvelabs.pegasus-1-5-v1:0` | `us.` (us-east-1, us-east-2, us-west-2) · `global.` | Video and a text prompt | 25 MB of base64 inline; up to 2 hours and under 2 GB from S3 | `maxOutputTokens` 512 to 98,304 | Video to text; time-coded segments |

IDs, profiles and the output range as of October 2026, measured in `us-east-1`; input
limits from the [Pegasus 1.5 model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html) page. The base
ID refuses on-demand calls, so every request goes through a profile. Unlike Pegasus 1.2,
1.5 has no `eu.` profile: from `eu-west-1` and `ap-northeast-1` it is reached through
`global.` (section 6).

Pegasus is on `bedrock-runtime` only and answers **InvokeModel** and
**InvokeModelWithResponseStream**. Converse refuses it.

**The video.** The notebook downloads [*Sparks*](https://opencontent.netflix.com/), a
short film from Netflix's Open Content programme, released under the
[Creative Commons Attribution 4.0 International](https://creativecommons.org/licenses/by/4.0/)
licence. It uses Netflix's smallest published encode (270p AV1, 8 MB, 3 minutes 49
seconds), which is 10.7 MB once base64-encoded and so under the 25 MB inline limit. That encode is a
video-only stream with no audio track, so the film's soundtrack is not part of it, and
the download cell checks this. Nothing below therefore exercises the audio path; point
the same requests at a video with a `soun` track to see what Pegasus does with speech.
The file is cached outside the repository and is not part of it. *Sparks* © Netflix, Inc., CC BY 4.0.

**Covered:** first call, streaming, structured output with a JSON Schema, reading text
on screen, time-coded segments with per-segment fields and what they cost, and the
`global.` profile from outside the US.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with boto3, and takes an availability check from
`_shared/bedrock.py`, this repository's own helper module.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import base64
import json
import re
import sys
import urllib.request
from pathlib import Path

import boto3
from botocore.config import Config

sys.path.insert(0, "../_shared")
from bedrock import endpoints_for

REGION = "us-east-1"
PEGASUS = "twelvelabs.pegasus-1-5-v1:0"
PROFILE = f"us.{PEGASUS}"

# A video call takes 15 to 35 seconds here, so allow well past boto3's 60-second default.
runtime = boto3.client("bedrock-runtime", region_name=REGION,
                       config=Config(read_timeout=300, retries={"max_attempts": 3}))

served = [name for name, ok in endpoints_for(PEGASUS, REGION).items() if ok]
print(f"{PEGASUS:30} served by: {', '.join(served) or 'neither endpoint in ' + REGION}")

twelvelabs.pegasus-1-5-v1:0    served by: runtime


### Download the video

The file comes from Netflix's public download bucket. The cell checks its size against
the published object, so a truncated download fails here rather than as a confusing
model error later. It also lists the file's track types: an MP4 names each track's kind
in a `hdlr` box, `vide` for video and `soun` for sound. It keeps the file under your
home directory so a re-run does not fetch it again.

In [2]:
VIDEO_URL = ("https://s3.amazonaws.com/download.opencontent.netflix.com/"
             "AV1/cmaf/spark-8b-59.94fps/spark_276kbps_270p.mp4")
VIDEO_BYTES = 8_029_782   # size of the published object, October 2026
INLINE_LIMIT = 25 * 1024 * 1024   # base64String limit from the Pegasus model card

cache = Path.home() / ".cache" / "sample-per-model-bedrock" / "sparks_270p.mp4"
if not cache.exists() or cache.stat().st_size != VIDEO_BYTES:
    cache.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(VIDEO_URL, timeout=120) as resp:  # noqa: S310  # nosec B310 - fixed https URL
        cache.write_bytes(resp.read())
video = cache.read_bytes()
if len(video) != VIDEO_BYTES:
    raise RuntimeError(f"expected {VIDEO_BYTES} bytes, got {len(video)}; delete {cache} and re-run")
tracks = {m.group(1).decode() for m in re.finditer(rb"hdlr\x00{8}(....)", video)}
if tracks != {"vide"}:
    raise RuntimeError(f"expected a single video track, found {sorted(tracks)}")
VIDEO_B64 = base64.b64encode(video).decode()
print(f"{cache.name}: {len(video):,} bytes, {len(VIDEO_B64):,} as base64, "
      f"under the inline limit: {len(VIDEO_B64) <= INLINE_LIMIT}")
print(f"track types: {sorted(tracks)}; has audio: {'soun' in tracks}")

sparks_270p.mp4: 8,029,782 bytes, 10,706,376 as base64, under the inline limit: True
track types: ['vide']; has audio: False


## 1. First call

The request body is Pegasus's own JSON: a prompt, the video under `mediaSource`, and
settings. `mediaSource` takes either `base64String`, as here, or an `s3Location` with the
object's `uri` and the `bucketOwner` account ID. The 25 MB inline limit applies to the
base64 string rather than to the file, and base64 adds a third, so inline tops out near
18 MB of video: a 20 MB file returns `ValidationException: Input is too long for
requested model.` (October 2026). From S3 the model takes up to 2 hours of video under
2 GB, which is the route for anything longer than a short clip. The answer is in
`message`, and `stopReason` is `stop` when it finished or `length` when it reached
`maxOutputTokens`.

The check looks for welding, which is what the worker in the film spends most of it
doing, rather than for exact words: wording changes between runs even at temperature 0.

In [3]:
def ask(prompt, model=PROFILE, client=runtime, **settings):
    body = {"inputPrompt": prompt, "mediaSource": {"base64String": VIDEO_B64},
            "temperature": 0, **settings}
    response = client.invoke_model(modelId=model, body=json.dumps(body),
                                   contentType="application/json", accept="application/json")
    return json.loads(response["body"].read())


first = ask("Describe what happens in this video in three sentences.")
print(first["message"])
print(f"\nstopReason={first['stopReason']}  input_tokens={first['input_tokens']}  "
      f"output_tokens={first['output_tokens']}")
print("mentions welding:", "weld" in first["message"].lower())

The video begins with a sunrise over a forest, transitioning to a construction worker preparing for his day by putting on safety gear. He is then shown welding steel beams, taking a break to drink water, and interacting with another worker. The video concludes with a water truck spraying the construction site.

stopReason=stop  input_tokens=153  output_tokens=59
mentions welding: True


## 2. Streaming

`invoke_model_with_response_stream` sends the answer in chunks, each with a `message`
fragment. The last chunk carries `stopReason` and Bedrock's invocation metrics.

In [4]:
body = {"inputPrompt": "Count the people who appear in this video, then describe each in "
                       "one short sentence.",
        "mediaSource": {"base64String": VIDEO_B64}, "temperature": 0}
stream = runtime.invoke_model_with_response_stream(
    modelId=PROFILE, body=json.dumps(body),
    contentType="application/json", accept="application/json")
parts, last = [], {}
for event in stream["body"]:
    if "chunk" in event:
        last = json.loads(event["chunk"]["bytes"])
        parts.append(last.get("message", ""))
        print(parts[-1], end="", flush=True)
metrics = last.get("amazon-bedrock-invocationMetrics", {})
print(f"\n\nstopReason={last.get('stopReason')}  chunks={len(parts)}  metrics={metrics}")

There

 are three people in the video:

 a construction worker welding

 in

 safety

...

 worker

 sweeping the floor, and a third

 worker in a red shirt talking

 to the first.



stopReason=stop  chunks=14  metrics={'inputTokenCount': 0, 'outputTokenCount': 36, 'invocationLatency': 15288, 'firstByteLatency': 3474}


## 3. Structured output with a JSON Schema

Set `responseFormat` to `{"type": "json_schema", "jsonSchema": {...}}`. Pegasus 1.5
requires the `type` field, which the Pegasus 1.2 request did not have. `message` then
holds a JSON string that matches the schema.

The check asserts that every scene it calls welding lies between 60 and 180 seconds,
where sections 1 and 2 put it.

In [5]:
SCENES = {
    "type": "object",
    "properties": {"scenes": {"type": "array", "items": {
        "type": "object",
        "properties": {"start_sec": {"type": "number"}, "end_sec": {"type": "number"},
                       "description": {"type": "string"}},
        "required": ["start_sec", "end_sec", "description"]}}},
    "required": ["scenes"],
}
answer = ask("List the scenes of this video.",
             responseFormat={"type": "json_schema", "jsonSchema": SCENES})
scenes = json.loads(answer["message"])["scenes"]
for s in scenes:
    print(f"{s['start_sec']:6.0f} - {s['end_sec']:4.0f} s  {s['description']}")

welding = [s for s in scenes if re.search(r"\bweld(s|ed|ing)\b", s["description"].lower())]
print(f"\nwelding scenes: {len(welding)}; inside 60-180 s: "
      f"{bool(welding) and all(60 <= s['start_sec'] and s['end_sec'] <= 180 for s in welding)}")

     0 -    8 s  Sunrise over a forested landscape.
     8 -   33 s  Construction worker in an elevator putting on safety gear.
    33 -   81 s  Worker walks through a construction site, climbs stairs, and prepares to weld.
    81 -  119 s  Worker welds a steel beam with sparks flying.
   119 -  142 s  Worker takes a break, drinks water, and talks to another worker.
   142 -  195 s  Worker sweeps the site and a truck drives away.

welding scenes: 1; inside 60-180 s: True


## 4. Text on screen

Pegasus reads writing in the frame as well as the action. The film ends on credits on a
black screen, so the cell asks for the title shown there and checks it against the one
the credits carry, `SPARKS`.

In [6]:
title = ask("What is the title of the film, as written on screen in the closing credits? "
            "Answer with the title only.")
print(repr(title["message"]), " stopReason:", title["stopReason"])
print("reads the on-screen title:", title["message"].strip().strip(".").upper() == "SPARKS")

'SPARKS'  stopReason: stop
reads the on-screen title: True


## 5. Time-coded segments

Pegasus 1.2 has no equivalent of this mode. Set `analysis_mode` to `time_based_metadata`
and describe each kind of moment in `segment_definitions`: an `id`, a `description` of
what to find, and a list of `fields`, each with a `name`, `type` and `description`, that
Pegasus fills in for every segment it finds. Each returned segment carries `start_sec`,
`end_sec`, one key per field name, and `_definition_id`, the `id` it matched. This is the
shape the [Pegasus 1.5 model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html) page documents. The service also accepts
`analysisMode` beside `responseFormat: {"type": "segment_definitions",
"segment_definitions": [...]}` and returns the same segments (October 2026).

This mode takes no prompt: adding `inputPrompt` returns a 400. It also reads far more of
the video. The cell prints the input tokens beside those of section 3's question on the
same file, and that ratio is what you pay for the timestamps.

In [7]:
DEFINITIONS = [
    {"id": "welding", "description": "The worker is welding a steel beam",
     "fields": [{"name": "sparks_visible", "type": "string",
                 "description": "Are sparks visible in this segment"}]},
    {"id": "break", "description": "The worker drinks water or talks with a colleague",
     "fields": [{"name": "who_else", "type": "string",
                 "description": "Who else appears in this segment"}]},
]
body = {"mediaSource": {"base64String": VIDEO_B64}, "temperature": 0,
        "analysis_mode": "time_based_metadata", "segment_definitions": DEFINITIONS}
response = runtime.invoke_model(modelId=PROFILE, body=json.dumps(body),
                                contentType="application/json", accept="application/json")
timed = json.loads(response["body"].read())
FIELDS = {d["id"]: [f["name"] for f in d["fields"]] for d in DEFINITIONS}
for seg in timed["segments"]:
    values = ", ".join(f"{name}={seg.get(name)!r}" for name in FIELDS[seg["_definition_id"]])
    print(f"{seg['_definition_id']:8} {seg['start_sec']:6.0f} - {seg['end_sec']:4.0f} s  {values}")
print("definition_errors:", timed["definition_errors"], " stopReason:", timed["stopReason"])

found = {seg["_definition_id"] for seg in timed["segments"]}
print(f"\nevery definition matched: {found == set(FIELDS)}")
print("every segment carries its fields: "
      f"{all(all(n in seg for n in FIELDS[seg['_definition_id']]) for seg in timed['segments'])}")
overlap = any(seg["_definition_id"] == "welding" and seg["start_sec"] < s["end_sec"]
              and s["start_sec"] < seg["end_sec"] for seg in timed["segments"] for s in welding)
print(f"welding segment overlaps section 3's welding scene: {overlap}")
print(f"input tokens: {timed['input_tokens']:,} here, {answer['input_tokens']:,} for section 3 "
      f"({timed['input_tokens'] / answer['input_tokens']:.0f}x)")

welding      81 -  139 s  sparks_visible='Yes, sparks are visible during the welding process.'
break       140 -  148 s  who_else='none'
break       149 -  162 s  who_else='colleague in red shirt'
definition_errors: []  stopReason: stop

every definition matched: True
every segment carries its fields: True
welding segment overlaps section 3's welding scene: True
input tokens: 94,561 here, 440 for section 3 (215x)


## 6. The `global.` profile from outside the US

Pegasus 1.5 has a `us.` profile and a `global.` one. From a Region outside the US, such as
`eu-west-1` or `ap-northeast-1`, only `global.` exists, and it may route the request to
any commercial Region. The base ID refuses on-demand calls in every Region.

In [8]:
eu = boto3.client("bedrock-runtime", region_name="eu-west-1",
                  config=Config(read_timeout=300, retries={"max_attempts": 3}))
reply = ask("In one sentence, what is the worker doing for most of the video?",
            model=f"global.{PEGASUS}", client=eu)
print(reply["message"])
print("mentions welding:", "weld" in reply["message"].lower())

try:
    ask("One sentence.", model=PEGASUS)
except runtime.exceptions.ValidationException as exc:
    print("\nbase ID:", str(exc).split(": ", 1)[1])

The worker is welding steel beams at a construction site for most of the video.
mentions welding: True



base ID: Invocation of model ID twelvelabs.pegasus-1-5-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.


## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Base model ID | 400 `ValidationException`: `Invocation of model ID twelvelabs.pegasus-1-5-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.` (October 2026) | Call `us.twelvelabs.pegasus-1-5-v1:0` or `global.twelvelabs.pegasus-1-5-v1:0`. |
| No `eu.` profile | Pegasus 1.2 has `eu.twelvelabs.pegasus-1-2-v1:0`; 1.5 has only `us.` and `global.` (October 2026) | From Europe or Asia Pacific use `global.`, and check whether sending video to any commercial Region is acceptable for your data. |
| Converse | 400 `ValidationException`: `This action doesn't support the model that you provided. Try again with a supported text or chat model.` | Use `InvokeModel` or `InvokeModelWithResponseStream` with Pegasus's own request body. |
| Response field names | The Pegasus 1.2 model card documents `message` and `finishReason`. 1.5 returns `message`, `stopReason`, `input_tokens` and `output_tokens`; in `time_based_metadata` mode it returns `segments`, `definition_errors`, `stopReason` and `finish_reason`. Each segment carries `start_sec`, `end_sec`, `_definition_id` and one key per declared field; a definition without `fields` returns its `description` there instead (October 2026) | Read `stopReason`, which both modes return, and read segment values by the field names you declared. |
| `responseFormat` without `type` | 400: `The responseFormat.type parameter is invalid. Field required`. An unknown type returns `Input should be 'json_schema' or 'segment_definitions'` | Send `{"type": "json_schema", "jsonSchema": {...}}`, the 1.2 shape plus `type`. |
| Output budget | `maxOutputTokens` 20 returns 400 `max_tokens must be at least 512`; 100000 returns `max_tokens must be at most 98304`. 512, 4096 and 8192 are accepted (October 2026). An answer cut off by the budget ends with `stopReason` `length` | Keep `maxOutputTokens` between 512 and 98,304, and treat `length` as an incomplete answer. The 1.2 limit of 4,096 does not apply. |
| Segments with a prompt | `analysis_mode=time_based_metadata does not accept prompt/prompt_v2 — use response_format.segment_definitions or custom_fields` | Leave `inputPrompt` out in that mode; put what you are looking for in each definition's `description`. |
| Segments without the mode | `response_format.type must be json_schema for analysis_mode=general` | Set `"analysis_mode": "time_based_metadata"` beside `segment_definitions`. The accepted modes are `general` and `time_based_metadata`; `analysisMode` is accepted too. |
| Segment definition fields | A definition-level `name` returns `Extra inputs are not permitted`, and a definition without `id` returns `Field required`; an empty list returns `segment_definitions is required (non-empty)` | Give each definition an `id`, a `description` and, for structured values per segment, `fields` of `name`, `type` and `description`. |
| Cost of timestamps | On this 3:49 film the segment request used about 94,000 input tokens across runs, against about 450 for the JSON Schema question in section 3, roughly 200 times as many (October 2026) | Use segments when you need the times, and plain questions for everything else. |
| Streaming token count | The last chunk's `amazon-bedrock-invocationMetrics` reports `inputTokenCount` 0, while the same request without streaming reports a few hundred `input_tokens` (October 2026) | Do not meter input from the streaming metrics; use the non-streaming fields or CloudWatch. |
| Temperature 0 | The same question returned different wording on each run (October 2026) | Check facts in the answer, such as a keyword or a time range, rather than its exact text. |
| Inline video size | The 25 MB limit on `base64String` ([model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html)) is on the encoded string, so about 18 MB of video. A 20 MB file returned `ValidationException: Input is too long for requested model.` (October 2026) | Send anything above about 18 MB from S3 with `mediaSource.s3Location` (`uri` and `bucketOwner`): up to 2 hours of video, under 2 GB. |

## Next

- Images rather than video: [`12-writer-palmyra/01`](../12-writer-palmyra/01-palmyra-vision.ipynb),
  [`13-amazon-nova/01`](../13-amazon-nova/01-nova-ladder-and-vision.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)